# Make fake training data

The first step in fine-tuning the models is to train them on fake data. Transcribing the images to data is difficult (that's what we are trying to train the models to do), but making images similar to the Daily Weather Records, given data is easy - we can write a python program to generate random daily rainfall values and generate images with the same structure s the real records. So we make 1000 or so fake images (where we know the data they contain) and then train the models using these images as training data. The capabilities learned on the fake data should carry over to the real data.

This notebook documents the process for making fake training images.

## Configure generation settings

How many to create, output directory, and the amount of variability to include in the images.

Note - the script to make the fake images is part of the weather-doc-extractor environment, so ideally we'd update the environment every time we made a change to that script. That's a terrible pain when debugging the script, so this notebook contains extra code to make sure the latest version of the code is always used.

In [ ]:
from pathlib import Path
import sys

# Ensure notebook imports use the local repository source tree.
# This avoids stale site-packages versions shadowing current edits.
_repo_root = Path.cwd().resolve()
if not (_repo_root / "src").exists():
    _repo_root = _repo_root.parent
_src_path = str(_repo_root / "src")
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)

# If an older installed package was already imported, clear it so imports reload from src.
for _name in list(sys.modules):
    if _name == "weather_doc_extractor" or _name.startswith("weather_doc_extractor."):
        del sys.modules[_name]

# Output root for generated synthetic data
OUTPUT_DIR = Path("../../fake_daily_rainfall")

# Main dataset size
N_RECORDS = 1000

# Reproducibility
SEED = 42

# Rendering parameters
FONT_SIZE = 20.0  # Typical font size
FONT_SIZE_JITTER = 0.22  # Amount of variation in the font size
JITTER_GRID_POINTS = 0.0008  # Amount of variation in the grid position
JPEG_QUALITY = 85  # JPEG quality for output images
RIGHT_DAY_LABEL_PROBABILITY = 0.9  # Most images have an additional column on the right with the day numbers
POST_DEC_BLANK_COLUMN_PROBABILITY = 0.2  # Some images have an additional blank column after that for Dec.
LINE_INTENSITY_SIGMA = 0.40  # Amount of variation in the image grid-line intensity
INDIVIDUAL_LINE_INTENSITY_SIGMA = 0.2  # Amount of variation in the individual grid-line intensity

OUTPUT_DIR

PosixPath('../fake_daily_rainfall')

## Generate synthetic dataset


In [43]:
import importlib

import weather_doc_extractor.make_fake_training_data.draw_grid as draw_grid_module
import weather_doc_extractor.make_fake_training_data.make_datasets as make_datasets_module

draw_grid_module = importlib.reload(draw_grid_module)
make_datasets_module = importlib.reload(make_datasets_module)
make_fake_dataset = make_datasets_module.main

make_fake_dataset(
    n_records=N_RECORDS,
    output_dir=str(OUTPUT_DIR),
    seed=SEED,
    font_size=FONT_SIZE,
    font_size_jitter=FONT_SIZE_JITTER,
    jitter_grid_points=JITTER_GRID_POINTS,
    jpeg_quality=JPEG_QUALITY,
    right_day_label_probability=RIGHT_DAY_LABEL_PROBABILITY,
    post_dec_blank_column_probability=POST_DEC_BLANK_COLUMN_PROBABILITY,
    line_intensity_sigma=LINE_INTENSITY_SIGMA,
    individual_line_intensity_sigma=INDIVIDUAL_LINE_INTENSITY_SIGMA,
)

## Verify counts and pairing

Checks that image and transcription stems match.

In [ ]:
images_dir = OUTPUT_DIR / "images"
transcriptions_dir = OUTPUT_DIR / "transcriptions"

image_stems = {p.stem for p in images_dir.glob("*.jpg")}
json_stems = {p.stem for p in transcriptions_dir.glob("*.json")}

n_images = len(image_stems)
n_json = len(json_stems)
n_paired = len(image_stems & json_stems)
n_unpaired_images = len(image_stems - json_stems)
n_unpaired_json = len(json_stems - image_stems)

print(f"Images: {n_images}")
print(f"Transcriptions: {n_json}")
print(f"Paired stems: {n_paired}")
print(f"Unpaired image stems: {n_unpaired_images}")
print(f"Unpaired transcription stems: {n_unpaired_json}")

## Build test dataset 

We're going to train models on the main fake dataset. We want to know if it worked, so make a small test dataset (using the same process) to validate the trained models against.

Creates `test_data/fake/{images,transcriptions}`. 
Same construction as main dataset, but smaller, and with different seed, so suitable for test & validation.

In [ ]:
import importlib

import weather_doc_extractor.make_fake_training_data.draw_grid as draw_grid_module
import weather_doc_extractor.make_fake_training_data.make_datasets as make_datasets_module

draw_grid_module = importlib.reload(draw_grid_module)
make_datasets_module = importlib.reload(make_datasets_module)
make_fake_dataset = make_datasets_module.main

make_fake_dataset(
    n_records=64,
    output_dir='../../test_data/fake',
    seed=SEED+1,
    font_size=FONT_SIZE,
    font_size_jitter=FONT_SIZE_JITTER,
    jitter_grid_points=JITTER_GRID_POINTS,
    jpeg_quality=JPEG_QUALITY,
    right_day_label_probability=RIGHT_DAY_LABEL_PROBABILITY,
    post_dec_blank_column_probability=POST_DEC_BLANK_COLUMN_PROBABILITY,
    line_intensity_sigma=LINE_INTENSITY_SIGMA,
    individual_line_intensity_sigma=INDIVIDUAL_LINE_INTENSITY_SIGMA,
)

## 6. Upload to Azure

In [ ]:
%%bash
bash ../scripts/aml_delete.sh fake_daily_rainfall/images
bash ../scripts/aml_upload.sh --src ../fake_daily_rainfall/images --dst fake_daily_rainfall/images
bash ../scripts/aml_delete.sh fake_daily_rainfall/transcriptions
bash ../scripts/aml_upload.sh --src ../fake_daily_rainfall/transcriptions --dst fake_daily_rainfall/transcriptions
bash ../scripts/aml_delete.sh test_data/fake/images
bash ../scripts/aml_upload.sh --src ../test_data/fake/images --dst test_data/fake/images
